# Portion 2 · TabBERT encoder: pretraining and handoffs

Pretrain the two-level TabBERT encoder with masked field modelling on train-period windows, then export embeddings, fine-tuning scores and reconstruction scores for Portions 3 and 4. Design: [`PORTION2_BUILD_GUIDE.md`](PORTION2_BUILD_GUIDE.md). Step-by-step plan: [`PORTION2_IMPLEMENTATION_PLAN.md`](PORTION2_IMPLEMENTATION_PLAN.md).

Run on a Colab GPU runtime (Runtime → Change runtime type → T4 GPU or better). Checkpoints and outputs go to Google Drive, not to Git.

## 1. Get the code and the Portion 1 data

The first code cell clones the repository and pulls only the Portion 1 files this portion reads through Git LFS (about 190 MB instead of the full 530 MB), which keeps LFS bandwidth low. When the notebook runs from a local checkout, it uses that checkout instead.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ted-cc6/TabBERT_AML"
BRANCH = "main"
LFS_FILES = [
    "Portion1/processed/event_tokens.npz",
    "Portion1/processed/window_index.csv.gz",
    "Portion1/processed/events_meta.csv.gz",
    "Portion1/processed/label_subsets.csv.gz",
    "Portion1/processed/label_subset_100pct.csv.gz",
]

current = Path.cwd()
PORTION2_DIR = next((p for p in (current, current / "Portion2") if (p / "portion2_encoder.py").is_file()), None)

if PORTION2_DIR is None:
    clone_root = Path("/content/aml_project_repo")
    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install", "--skip-smudge"], check=True)
    if not clone_root.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(clone_root)], check=True)
    else:
        subprocess.run(["git", "-C", str(clone_root), "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", str(clone_root), "lfs", "pull", "--include", ",".join(LFS_FILES)], check=True)
    PORTION2_DIR = clone_root / "Portion2"

PROCESSED_DIR = PORTION2_DIR.parent / "Portion1" / "processed"
sys.path.insert(0, str(PORTION2_DIR))
for name in LFS_FILES:
    size = (PORTION2_DIR.parent / name).stat().st_size
    assert size > 1_000_000, f"{name} is still an LFS pointer ({size} bytes)"
print(PORTION2_DIR)

## 2. Check the GPU and mount Drive

`OUT_DIR` is the shared Drive folder for checkpoints, metrics, embeddings and score files. Each run writes to `OUT_DIR/<run_name>/`.

In [ ]:
import torch

import portion2_encoder as enc

DEVICE = enc.pick_device()
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type != "cuda":
    print("No CUDA GPU: switch the runtime to a GPU before the full run.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/TabBERT_AML/portion2_runs")
except ImportError:
    OUT_DIR = PORTION2_DIR / "runs"  # local runs; ignored by Git
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUT_DIR)

## 3. Main pretraining, seed 42 (Task 4)

Field-level masking with Δt, 1M train windows per epoch, 3 epochs, batch 512, AdamW 1e-3 with warmup and cosine decay. Progress prints every 200 steps. The run ends with a validation pass per epoch and writes `checkpoint.pt`, `metrics.csv` and `amount_embedding_cosine.png`.

Set `QUICK_CHECK = True` first to confirm the runtime works on 50K windows (about a minute), then set it back to `False` for the real run.

In [ ]:
import dataclasses
import time

QUICK_CHECK = True

cfg = enc.load_run_config(PORTION2_DIR / "configs" / "main_seed42.json")
if QUICK_CHECK:
    cfg = dataclasses.replace(cfg, run_name="quick_check", windows_per_epoch=50_000, epochs=1, valid_windows=5_000, warmup_steps=20)
print(cfg)

start = time.time()
ckpt_path = enc.pretrain(cfg, PROCESSED_DIR, OUT_DIR, device=DEVICE, log_every=200)
print(f"Done in {(time.time() - start) / 60:.1f} min:", ckpt_path)

## 4. Check the result

Done when each field's masked accuracy (`acc_`) is clearly above its most-frequent-token baseline (`baseline_`), especially for currency, counterparty bank, amount and time gap. Direction at the last position is excluded, because the target event is always "sent". The heatmap should show neighbouring amount buckets as similar (a bright band along the diagonal).

In [ ]:
import pandas as pd
from IPython.display import Image, display

metrics = pd.read_csv(ckpt_path.parent / "metrics.csv")
fields = [s.name for s in enc.load_field_specs(PROCESSED_DIR, cfg.use_dt)]
summary = pd.DataFrame({
    "accuracy": [metrics[f"acc_{f}"].iloc[-1] for f in fields],
    "baseline": [metrics[f"baseline_{f}"].iloc[-1] for f in fields],
}, index=fields)
summary["lift"] = summary.accuracy - summary.baseline
display(metrics[["epoch", "train_loss", "valid_loss", "seconds"]])
display(summary.round(3))
extra = [c for c in metrics.columns if c.startswith("acc1_") or c.endswith("_nonzero")]
display(metrics[extra].round(3))
display(Image(filename=str(ckpt_path.parent / "amount_embedding_cosine.png")))